# RAG workshop: an AI that looks things up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hugobb/IndigenousAI/blob/main/notebooks/rag_workshop_beginner.ipynb)

An AI chatbot answers from what it learned during its training. RAG (Retrieval-Augmented Generation) lets it look things up first. RAG finds the passages of a document that match your question, and the AI writes its answer from them.

In this notebook you try each step of RAG on one Wikipedia article. You don't need to know how to code. The grey boxes are cells. To run a cell, click the ▶️ button on its left. Go from top to bottom.

## 0. Setup (5 min)

**Do this:**
1. In the menu at the top, click **Runtime**, then **Change runtime type**. Choose **T4 GPU** and click **Save**.
2. Click ▶️ on the **Setup** cell below.
3. If a message says "This notebook was not authored by Google", click **Run anyway**.
4. Wait until the cell shows **✅ Ready**. This takes 1 to 2 minutes.

If the cell shows ⚠️ instead, everything still works, only slower. Tell an assistant.

If a cell later shows an error that ends with "is not defined", run the Setup cell again, then that cell.

In [ ]:
#@title Setup: click ▶️ and wait for ✅ Ready (1 to 2 minutes) { display-mode: "form" }
# You don't need to read this cell. It installs what is missing, defines the helpers the
# workshop uses (ask, similarity, sections, split, where_is, search, ask_with_documents)
# and loads the two AI models. Curious? All the code is below.
import logging
import warnings

# Hide two harmless notices seen outside Colab: tqdm's progress bars, and a macOS note from torch.
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip install -q sentence-transformers transformers accelerate

import os
import re
from collections.abc import Iterator
from dataclasses import dataclass
from queue import Empty
from threading import Thread
from typing import Literal
import numpy as np


# ======================================================================
# content.py
# ======================================================================
"""Everything the notebook works with: the article, the example sentences and the example questions.

The article is a shortened copy of the Wikipedia article "2026 FIFA World Cup final" (a pinned
revision, CC BY-SA 4.0), saved by fetch_article.py in documents/2026_world_cup_final.txt.
The example values were picked with find_demo_values.py. tests/test_effects.py (embeddings and
search) and tests/test_llm_effects.py (the two Qwen models) check that each one does what the
notebook text says.
"""


def _load(name: str) -> str:
    """Read a file from documents/. When the notebook is built, each call is replaced by the text itself."""
    from pathlib import Path

    return (Path(__file__).resolve().parent / "documents" / name).read_text(encoding="utf-8")


# ---------------------------------------------------------------- the article
_HEADER, _BODY = 'Title: 2026 FIFA World Cup final (Wikipedia)\nSource: https://en.wikipedia.org/w/index.php?title=2026_FIFA_World_Cup_final&oldid=1385698804\nRetrieved: 2026-11-20\nLicense: CC BY-SA 4.0, https://creativecommons.org/licenses/by-sa/4.0/\nShortened: some sections removed.\n\nThe final match of the 2026 FIFA World Cup—the 23rd edition of the FIFA competition for men\'s national soccer teams—was played at MetLife Stadium in East Rutherford, New Jersey, part of the New York metropolitan area, on July 19, 2026. It was contested by Spain and the defending champion Argentina.\n\nThe tournament was contested by hosts Canada, Mexico, and the United States, along with 45 other teams who emerged from the qualification phase, which was organized by the six FIFA confederations. The 48 teams competed in a group stage, from which 32 teams qualified for the knockout stage. En route to the final, Spain finished first in Group H with two wins and a draw, before defeating Austria in the round of 32, Portugal in the round of 16, Belgium in the quarterfinal, and France in the semifinal. Argentina finished first in Group J with three wins, after which it defeated Cape Verde in the round of 32, Egypt in the round of 16, Switzerland in the quarterfinal, and England in the semifinal. The final took place in front of 80,663 spectators, and was refereed by Slavko Vinčić of Slovenia.\n\nSpain dominated possession and created numerous goal-scoring opportunities throughout the match, with Argentina goalkeeper Emiliano Martínez repeatedly denying its offensive efforts. Argentina, reduced to ten players after Enzo Fernández was sent off in the second half, failed to register a single shot in regulation time, a first for a World Cup final. The goalless deadlock carried into extra time, during which Ferran Torres scored from a Nico Williams header to secure a 1–0 victory for Spain. With a few minutes left, Argentina missed multiple opportunities to equalize the score. After the match, a brawl broke out between the two teams.\n\nSpain\'s win earned the country its second World Cup title and its first since 2010. Torres was named the man of the match, and Emiliano Martínez\'s 11 saves set a record for the most saves by a goalkeeper in a World Cup final. The match was the first ever World Cup final to include a halftime show, which caused the full halftime break to last over 27 minutes, against International Football Association Board (IFAB) protocol. Spain became the first country to simultaneously hold the men\'s and women\'s World Cup titles, having won the FIFA Women\'s World Cup in 2023.\n\n== Background ==\n\nThe 2026 FIFA World Cup was the 23rd edition of the World Cup, FIFA\'s soccer competition for men\'s national teams, held in Canada, Mexico and the United States between June 11 and July 19, 2026. The tournament featured 48 teams for the first time, up from 32 in the 2022 World Cup. The national teams of Canada, Mexico and the United States qualified automatically for the tournament as hosts, while 206 teams competed for the remaining 45 slots through qualifying rounds organized by the six FIFA confederations and held between September 2023 and March 2026. In the tournament, the teams were divided into twelve groups of four, with each team playing every other team once in a round-robin format. The top two teams from each group, along with the top eight third-place teams, advanced to the knockout stage.\n\nArgentina had won the previous World Cup in Qatar in 2022, defeating France in a penalty shoot-out in the final after the match finished 3–3 after extra time. Argentina had previously won the World Cup in 1978 and 1986, and finished as runners-up in 1930, 1990, and 2014. Argentina manager Lionel Scaloni was attempting to become the second manager to win multiple World Cup titles, after Vittorio Pozzo of Italy in 1934 and 1938. Spain had won the World Cup once before in 2010, defeating the Netherlands in the final in Johannesburg, South Africa. Heading into the 2026 final, Spain was tied with Italy for the longest undefeated streak in the history of men\'s international soccer, with 37 matches.\n\nSpain and Argentina had played each other fourteen times since 1952, with each team winning six matches. They had played one World Cup match before, in the group stage of the 1966 tournament, with Argentina winning 2–1. Their most recent meeting before the 2026 World Cup was a friendly match in Madrid on March 27, 2018, which Spain won 6–1. Spain and Argentina, respective winners of the UEFA European Championship and Copa América in 2024, were scheduled to contest the 2026 Finalissima in Qatar in March 2026. However, the match was cancelled due to regional security concerns surrounding the Iran war.\n\nThe 2026 final was the first World Cup final to feature two reigning continental champions, who were the two top-ranked teams in the FIFA World Ranking. It was also the second time that two Spanish-speaking countries competed against each other, after Uruguay and Argentina contested the inaugural 1930 final. With Spain and Argentina co-hosting the next World Cup in 2030, it meant the winner of this final would be the first-ever team to host a World Cup as the defending champion. The official match ball used in the semifinals, match for third place, and the final, was a variation of the Adidas Trionda called the Trionda Final. It has the same technical specifications as the Adidas Trionda, but has a gold, white, and black design that resembles the FIFA World Cup Trophy.\n\n== Route to the final ==\n\n=== Spain ===\n\nSpain qualified for the 2026 World Cup after finishing first in its qualification group, and entered the tournament as the bookmakers\' favorite to win. Spain was drawn into Group H, alongside Cape Verde, Saudi Arabia, and Uruguay. Its first match was against Cape Verde in Atlanta on June 15. The game ended in a goalless draw, due to a strong defensive performance from Cape Verde and its goalkeeper, Vozinha. Spain\'s second match was against Saudi Arabia on June 21. Lamine Yamal gave Spain the lead ten minutes into the game, scoring his first World Cup goal from a cross by Mikel Oyarzabal. Oyarzabal added a second goal from a corner kick taken by Yamal, and within three minutes he scored again from Dani Olmo\'s header to give Spain a 3–0 halftime lead. In the second half, a volley by Spain\'s Marc Cucurella was saved by Saudi Arabian goalkeeper Hassan Al Tambakti, but it deflected off Hassan Al Tambakti for an own goal to seal Spain\'s 4–0 victory. Spain\'s last group game was against Uruguay in Guadalajara on June 26. Spain won 1–0 after Álex Baena\'s shot slipped through the hands of Uruguay goalkeeper Fernando Muslera. Spain advanced to the knockout stage as group winners with seven points. In the round of 32, Spain faced Austria at SoFi Stadium in Inglewood, California on July 2. Oyarzabal gave Spain a 1–0 lead in the 38th minute, scoring from a cross by Cucurella. In the second half, Pedro Porro scored Spain\'s second goal when he headed the ball from a cross by Baena. One minute before the end of regulation time, Oyarzabal added a third goal from a cross by Cucurella to seal a 3–0 win for Spain. Spain\'s victory was its first in a World Cup knockout stage match since the 2010 final.\n\n=== Argentina ===\n\nArgentina earned its World Cup berth by placing first in South American qualifiers, and entered the tournament also as one of the favorites to win. Argentina was drawn into Group J, alongside Algeria, Austria, and Jordan. In its first match against Algeria at Arrowhead Stadium in Kansas City on June 16, Argentina won 3–0 through a hat trick by Lionel Messi. Their second match was against Austria at AT&T Stadium on June 22. Argentina took the lead in the first half with a goal by Messi, who surpassed Miroslav Klose to become the all-time top goalscorer at World Cup tournaments, with 17 goals. Messi added a second goal in second-half injury time to secure Argentina a 2–0 victory. Argentina\'s last group game was against Jordan on June 27. Argentina won a free kick in the 19th minute, when Mohannad Abu Taha fouled Giovani Lo Celso. Celso took the kick and scored to give Argentina a 1–0 lead. After Nizar Al-Rashdan of Jordan fouled Argentina\'s Marcos Senesi, Lautaro Martinez scored from the penalty kick in the 30th minute, giving Argentina a 2–0 lead. In the second half, Musa Al-Taamari, who had come on as a substitute, scored once for Jordan to make the score 2–1. Messi took a free kick for Argentina after being fouled, and scored with a curling shot to give Argentina a 3–1 win. Argentina qualified as group winners with nine points.\n\n== Match ==\n\n=== First half ===\n\nArgentina kicked off the match at 3:06 p.m. EDT (7:06 p.m. UTC) in front of a crowd of 80,663. The weather at Teterboro Airport, 4.4 mi (7.1 km) from the stadium, was recorded as fair before kick-off, with a temperature of 80 °F (27 °C) and 40% humidity. Five minutes into the game, Lamine Yamal had a chance to score for Spain when he received a pass from Dani Olmo. He took a shot that deflected off Lisandro Martinez\'s foot and was saved by Argentina\'s goalkeeper, Emiliano Martínez. Julián Álvarez of Argentina attempted to pass the ball to Lionel Messi, but Spain\'s goalkeeper Unai Simón ran 40 yards (36.5 m) from the goal to outpace him and kick it away. In the 11th minute, Yamal attempted to receive the ball from a cross by Rodri, but it went out of the touchline. A minute later, Spain won a corner kick after Fabián Ruiz\'s shot was deflected by Emiliano Martínez in the penalty area.\n\nTwo minutes later, Alexis Mac Allister of Argentina fouled Dani Olmo. In the 15th minute, Argentina won a free kick 45 yards (41.1 m) from Spain\'s goal after Messi was tackled by Alex Baena. Rodrigo De Paul\'s kick went over the wall and was saved by Simón. In the 39th minute, Spain\'s Mikel Oyarzabal received a pass from Fabián Ruiz outside of the penalty area and made a 20-yard (18 m) low shot at the goal, but it was saved by Emiliano Martínez. In the 41st minute, Lisandro Martínez was given a yellow card for a foul on Oyarzabal, and a minute later a left-footed shot from Marc Cucurella went wide of the goal. Argentina was forced to make a substitution after Lisandro Martínez became injured, with Nicolás Otamendi taking his place.\n\n=== Second half ===\n\nThe second half began with a substitution from Argentina, with Leandro Paredes replacing Nicolás Gonzalez. One minute later, Baena made a 22-yard (20 m) shot towards Argentina\'s goal which was saved by Emiliano Martínez. Paredes received a yellow card for pushing Rodri. In the 57th minute, Argentina brought Nahuel Molina onto the pitch in place of Gonzalo Montiel. In the 60th minute, Spain won a free kick when Paredes fouled Pedro Porro 25 yards (23 m) from Argentina\'s goal. Yamal took the kick and hit the wall. Spain then made two substitutions, with Pedri and Ferran Torres coming on in place of Ruiz and Oyarzabal, respectively. In the 64th minute, Olmo hit a 22-yard (20 m) shot that deflected off Martinez and went wide of the goalpost. Three minutes later, Spain had what Chris Sutton described as a "golden opportunity" when Yamal delivered a cross to Torres, who hit a goal-bound header that was saved by Martínez. In the 70th minute, Argentina made two substitutions when Giuliano Simeone and Facundo Medina replaced De Paul and Cristian Romero, respectively. Five minutes later, Spain sent Nico Williams and Mikel Merino onto the pitch in place of Baena and Olmo. In the 77th minute, Pedri made a shot from the edge of the penalty area, but it was saved by Martínez. Cubarsí then struck a 25-yard (23 m) shot at Argentina\'s goal, which Martinez saved with a dive. In the 80th minute, Nico Williams made a cross that Aymeric Laporte tried to meet with a header, but Martínez caught the ball. In the 82nd minute, Argentina\'s Enzo Fernández fell to the ground following contact from Rodri, and the referee did not award a foul, leading Fernández to complain and then receive a yellow card for dissent. In the 3rd minute of stoppage time, Argentina\'s Giuliano Simeone made a low cross from the right, but Unai Simón intercepted the ball. On the resulting counterattack, Williams dribbled the ball for 30 meters before taking a shot from the edge of the box, which was also saved by Martínez. A few minutes later, Fernández was sent off after receiving a second yellow card for tackling Pau Cubarsí while attempting to win a loose ball; a VAR review upheld the on-field decision. In the final seconds of regulation time, Yamal struck a free kick that Martínez managed to deflect away. As regulation time concluded, Argentina became the first team to fail to register a shot during the first 90 minutes of a World Cup final.\n\n=== Extra time ===\n\nIn the 93rd minute of the match during extra time, Williams was not able to head a Pedri cross with enough power, and Martínez dove to his right to make the save. Three minutes later, Williams appeared to score, but the goal was disallowed because Merino had stepped on Otamendi\'s foot. Commentators noted that the contact was minor and was the type that referees sometimes overlook in the penalty area, and also observed that Otamendi had his arm around Merino prior to the incident. Spain supporters sarcastically chanted "Infantino" following the disallowed goal, referencing allegations of FIFA favoring Argentina.\n\nMerino then headed the ball wide of Argentina\'s goal in the 102nd minute. At the start of the second period of extra time, Torres received a ball headed by Williams. He struck it with his left foot from close range and past Martínez, scoring the first goal of the match and putting Spain in the lead 1–0.\n\nAfter Spain\'s goal, Argentina began to counterattack in search of an equalizer goal. In the 113th minute, Torres appeared to score his second goal, but it was ruled out for being offside. Two minutes later, Messi attempted to catch Simón off guard with a cross that drifted toward the Spanish goal, passing over the crossbar. In the 116th minute, Williams positioned himself in Argentina\'s box after outrunning Nahuel Molina, but was unable to connect with the ball to shoot. A minute later, Messi made Argentina\'s first shot from the edge of the box, which struck Merino in the face as he blocked it, leaving him down on the pitch.\n\nIn the 120th minute, Argentina\'s final attack came from a Giuliano Simeone pass toward Marcos Senesi in the box, but Cubarsí cleared the ball. The resulting corner kick led to a scramble inside the goal area, with the Spanish defense kicking the ball out of bounds. The next corner kick was deflected by Argentina\'s Nicolás Tagliafico on the near post and fell to Simeone\'s feet, but his shot from close to the penalty spot went well over the crossbar. In the final minutes, Spain kept possession away from its goal. A long free kick from Emiliano Martinez into Spain\'s goal area was cleared by its defense before the final whistle confirmed Spain\'s victory.\n\n== Post-match ==\n\nSpain\'s Ferran Torres, who scored the only goal of the final, was named as the man of the match. In the awards ceremony after the match, Spain\'s Rodri was awarded the Golden Ball as the best player of the tournament, while his teammate Unai Simón was awarded the Golden Glove as the best goalkeeper. Spain\'s Pau Cubarsí won the FIFA Young Player Award as the best young player of the tournament. Lionel Messi won the Silver Ball as the second-best player of the tournament, and the Silver Boot as the second-top goalscorer, with eight goals. FIFA president Gianni Infantino and US president Donald Trump were booed by some spectators as they entered the field with the World Cup trophy, which they presented to Spain\'s captain Rodri.\n\nLamine Yamal and Messi embraced after the match, a moment described by FIFA and several media outlets as symbolic of a generational transition in football, with Messi approaching the end of an illustrious career and Yamal, regarded as one of the sport\'s most promising talents, at the start of his. The encounter drew additional resonance from a 2007 photograph in which Messi, then 20 years old, bathed the infant Yamal during a charity photo session while at FC Barcelona. The image had attracted renewed media attention as the two players prepared to face each other in the 2026 World Cup final.\n\nSpain became the first country to simultaneously hold both the men\'s and women\'s World Cup titles, having won the Women\'s World Cup in 2023. Spain also became the second European nation to win a World Cup held in the Americas, following Germany in 2014. Messi became the second player in history to play in three World Cup finals, after Cafu of Brazil. The 2026 final featured 11 saves by Argentina goalkeeper Emiliano Martínez, the most ever recorded in a World Cup final.\n'.split("\n\n", 1)
SOURCE = dict(line.split(": ", 1) for line in _HEADER.splitlines())  # Title, Source, Retrieved, License, Shortened
ARTICLE = _BODY.strip()  # the article without its 5-line attribution block
ARTICLE_URL = SOURCE["Source"]
ARTICLE_RETRIEVED = SOURCE["Retrieved"]
ARTICLE_REVISION = ARTICLE_URL.rsplit("=", 1)[1]

# ---------------------------------------------------------------- 1. The problem: the AI alone
NO_DOC_QUESTION = "How long did the halftime break last in the 2026 World Cup final?"
NO_DOC_ANSWER = "27"  # "over 27 minutes": in the AI's answer with the article, not without it
WINNER_QUESTION = "Who won the 2026 FIFA World Cup final?"
WINNER = "Spain"
# Section 5 asks again, with the article, in this short form: both models answer it. With "in the 2026
# World Cup final" added, Qwen3-0.6B said "I don't know" even with the right chunk (pre-test, 2026-11-20).
WITH_DOC_QUESTION = "How long did the halftime break last?"
NO_DOC_ANSWER_IN_ARTICLE = "halftime break to last over 27 minutes"

# ---------------------------------------------------------------- 2. Embeddings
SAME_MEANING = ("Spain won the final.", "The Spanish team became world champions.")
SAME_WORD = ("Spain won the final.", "This is my final offer.")
UNRELATED = ("Spain won the final.", "I had soup for lunch.")
LOVE_HATE = ("I love winter.", "I hate winter.")

# ---------------------------------------------------------------- 3. Search, over the 8 sections
HIT_QUESTION = "Which awards did Spain's players win?"
HIT_SECTION = "Post-match"
MISS_QUESTION = "Who was the referee of the final?"
MISS_SECTION = "Introduction"  # where the answer is; search does not put it in the top 3
MISS_ANSWER = "Slavko Vinčić"
NONSENSE_QUESTION = "What is the best pizza topping?"

# ---------------------------------------------------------------- 4. Splitting
DEMO_SIZE = 50  # words per chunk in sections 4 to 6
DEMO_OVERLAP = 15
TINY_SIZE = 5
SPLIT_QUESTION = "Why was the goal by Williams disallowed?"
CUT_PHRASE = "the goal was disallowed because Merino had stepped on Otamendi's foot"  # cut at DEMO_SIZE words
SPLIT_ANSWER = "Otamendi"  # in the AI's answer with DEMO_OVERLAP, not without overlap

# ---------------------------------------------------------------- 5. Answering
TOP1_VS_TOP3_QUESTION = "Where was the final played?"
TOP1_VS_TOP3_ANSWER = "MetLife"  # in the AI's answer with top=3, not with top=1
TOP1_VS_TOP3_ANSWER_IN_ARTICLE = "played at MetLife Stadium"  # in the 2nd best chunk, not the 1st
# Our shortened copy mentions the halftime show but not who performed: fetch_article.py drops the full
# article's "Halftime show" section, which names the performers.
OUT_OF_DOC_QUESTION = "Who performed at the halftime show?"
OUT_OF_DOC_ABSENT = ["performed", "performer", "sang", "singer", "concert"]

# ---------------------------------------------------------------- 6. Wrap-up
WRAP_UP_QUESTION = MISS_QUESTION  # the question search missed in section 3, now with the whole pipeline
WRAP_UP_ANSWER = "Vinčić"


# ======================================================================
# toolkit.py
# ======================================================================
"""The workshop's machinery: chunking, similarity, search, prompts and the two AI models."""



Strategy = Literal["fixed", "overlap", "sentences", "whole"]


# ---------------------------------------------------------------- chunking
def split_sentences(text: str) -> list[str]:
    """Cut text after each '.', '?' or '!'."""
    # Split after . ? ! followed by a capital letter or digit, but not after a lone capital ("E. coli").
    pattern = r"(?<=[.?!])(?<!\b[A-Z]\.)\s+(?=[A-Z0-9\"“‘(])"
    return [s.strip() for s in re.split(pattern, text) if s.strip()]


def chunk_fixed(text: str, size: int) -> list[str]:
    """Cut text into pieces of `size` words, one after the other."""
    words = text.split()
    return [" ".join(words[i : i + size]) for i in range(0, len(words), size)]


def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Like chunk_fixed, but each piece repeats the last `overlap` words of the previous one."""
    overlap = max(0, min(overlap, size - 1))
    step = size - overlap
    words = text.split()
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i : i + size]))
        if i + size >= len(words):
            break
    return chunks


def chunk_sentences(text: str, n: int) -> list[str]:
    """Group text into pieces of `n` sentences."""
    sentences = split_sentences(text)
    return [" ".join(sentences[i : i + n]) for i in range(0, len(sentences), n)]


def make_chunks(text: str, strategy: Strategy, size: int, overlap: int) -> list[str]:
    """One entry point for every chunking strategy. For 'sentences', size = sentences per chunk."""
    if strategy == "fixed":
        return chunk_fixed(text, size)
    if strategy == "overlap":
        return chunk_overlap(text, size, overlap)
    if strategy == "sentences":
        return chunk_sentences(text, size)
    return [text.strip()]


# ---------------------------------------------------------------- similarity and search
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def similarity_label(score: float) -> str:
    """Plain-language reading of a similarity score."""
    if score >= 0.60:
        return "very similar"
    if score >= 0.40:
        return "somewhat related"
    if score >= 0.20:
        return "loosely related"
    return "unrelated"


def rank(query_vec: np.ndarray, doc_vecs: np.ndarray, k: int) -> list[tuple[int, float]]:
    """The k stored texts closest to the query, best first, as (position, score)."""
    scores = [cosine(query_vec, d) for d in doc_vecs]
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, scores[i]) for i in order[: min(k, len(scores))]]


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


class MiniLMEmbedder:
    """Turns text into embeddings (384 numbers each) with a small, fast model."""

    def __init__(self, device: str):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(EMBED_MODEL, device=device)

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


# ---------------------------------------------------------------- prompt
def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


# ---------------------------------------------------------------- map of meaning
def project_2d(vectors: np.ndarray) -> np.ndarray:
    """Squash many-number embeddings down to 2 numbers each (PCA), so they can be drawn on a map."""
    x = np.asarray(vectors, dtype=float)
    x = x - x.mean(axis=0)
    out = np.zeros((len(x), 2))
    if len(x) < 2:
        return out
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    proj = x @ vt[:2].T
    out[:, : proj.shape[1]] = proj
    return out


# ---------------------------------------------------------------- the answer-writing model
CHAT_MODEL_GPU = "Qwen/Qwen3-1.7B"
CHAT_MODEL_CPU = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 200


def model_dtype(device: str):
    """Full precision on the GPU: half precision (float16) can overflow with Qwen3 on a T4."""
    import torch

    return torch.float32 if device == "cuda" else torch.bfloat16


def dtype_argument(device: str) -> dict:
    """The precision argument for from_pretrained. transformers 4.56 renamed torch_dtype to dtype:
    older versions crash on dtype=, newer ones print a warning for torch_dtype=."""
    import transformers

    major, minor = (int(part) for part in transformers.__version__.split(".")[:2])
    return {"dtype" if (major, minor) >= (4, 56) else "torch_dtype": model_dtype(device)}


def stream_worker(target, streamer) -> Iterator[str]:
    """Run `target` (the model writing) in the background and pass on its text as it arrives.
    If it fails, stop waiting and raise instead of hanging forever."""
    failure: list[BaseException] = []

    def work():
        try:
            target()
        except BaseException as error:  # noqa: BLE001 - any failure must reach the participant
            failure.append(error)
            streamer.text_queue.put(streamer.stop_signal)

    worker = Thread(target=work, daemon=True)
    worker.start()
    try:
        yield from streamer
    except Empty:
        raise RuntimeError("The model took too long to answer.") from None
    worker.join(timeout=1)
    if failure:
        raise RuntimeError(f"The model stopped with an error: {failure[0]}") from failure[0]


class QwenGenerator:
    """Writes answers with a small Qwen model. Greedy decoding: the same question gives the same answer."""

    def __init__(self, model_name: str, device: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **dtype_argument(device)).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=180)
        settings = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
        yield from stream_worker(lambda: self.model.generate(**settings), streamer)


class FakeGenerator:
    """A stand-in for tests: repeats the first passage instead of thinking."""

    def generate(self, prompt: str) -> Iterator[str]:
        yield "(test answer) "
        first = next((line for line in prompt.splitlines() if line.startswith("[1] ")), prompt)
        for word in first.removeprefix("[1] ").split():
            yield word + " "


@dataclass
class Models:
    embedder: MiniLMEmbedder
    generator: QwenGenerator | FakeGenerator
    device: str
    llm_name: str


MODELS: Models | None = None
NOT_READY = "Run the Setup cell first."


def load_models() -> Models:
    """Load both models, on the GPU when there is one. The result is also kept in MODELS."""
    global MODELS
    MODELS = _load_models()
    return MODELS


def _load_models() -> Models:
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = MiniLMEmbedder(device)
    if os.environ.get("RAG_WORKSHOP_FAKE_LLM") == "1":
        return Models(embedder, FakeGenerator(), device, "fake")
    name = os.environ.get("RAG_WORKSHOP_LLM") or (CHAT_MODEL_GPU if device == "cuda" else CHAT_MODEL_CPU)
    return Models(embedder, QwenGenerator(name, device), device, name)


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def status_line(device: str, in_colab: bool | None = None) -> str:
    """The Setup cell's last line. Only Colab has a Runtime menu that can switch on a GPU."""
    if device == "cuda":
        return "✅ Ready. You can go on to section 1."
    if in_colab is None:
        in_colab = running_in_colab()
    slow = "⚠️ Ready, but there is no GPU, so the AI is slower (up to a minute per answer) and makes more mistakes. You can carry on."
    colab = " To get a GPU: click Runtime > Change runtime type > T4 GPU > Save, then click ▶️ on this cell again."
    return slow + (colab if in_colab else "")


def answer(models: Models, question: str, contexts: list[str], allow_unknown: bool) -> Iterator[str]:
    """The whole RAG step: build the prompt from the retrieved passages and stream the model's answer."""
    return models.generator.generate(build_prompt(question, contexts, allow_unknown))


# ---------------------------------------------------------------- plain helpers for the notebook cells
SHOW = 60  # words of a passage shown on screen: a whole chunk of up to 60 words, the start of a longer one


def _short(text: str) -> str:
    words = text.split()
    return " ".join(words[:SHOW]) + (" ..." if len(words) > SHOW else "")


def _embed(texts: list[str]) -> np.ndarray:
    return MODELS.embedder.embed(texts)


def score(a: str, b: str) -> float:
    """How close in meaning two texts are (cosine of their embeddings)."""
    if MODELS is None:
        print(NOT_READY)
        return 0.0
    va, vb = _embed([a, b])
    return cosine(va, vb)


def similarity(a: str, b: str) -> None:
    """Print how similar two texts are, e.g. 'Similarity: 0.83'."""
    if MODELS is None:
        print(NOT_READY)
        return
    print(f"Similarity: {score(a, b):.2f}")


def split(document: str, words_per_chunk: int = 60, overlap: int = 0) -> list[str]:
    """Cut a document into chunks of words. 0 words means one chunk for everything."""
    if words_per_chunk <= 0:
        chunks = [document.strip()] if document.strip() else []
    else:
        chunks = chunk_overlap(document, words_per_chunk, max(0, overlap))
    print(f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'}.")
    return chunks


_ATTRIBUTION = ("Title:", "Source:", "Retrieved:", "License:", "Shortened:")
_HEADING = re.compile(r"^(={2,3})\s*(.+?)\s*\1\s*$")


def sections(document: str) -> list[str]:
    """One passage per section (== and === headings). The heading path is the first line."""
    lines = document.splitlines()
    head = []
    for line in lines:
        if not line.strip():
            break
        head.append(line)
    if head and all(line.startswith(_ATTRIBUTION) for line in head):
        lines = lines[len(head) :]

    passages: list[str] = []
    title = "Introduction"
    parent = ""
    body: list[str] = []

    def flush() -> None:
        text = "\n".join(body).strip()
        if text:
            passages.append(f"{title}\n{text}")
        body.clear()

    for line in lines:
        match = _HEADING.match(line.strip())
        if not match:
            body.append(line)
            continue
        flush()
        name = match.group(2)
        if len(match.group(1)) == 2:
            parent = name
            title = name
        else:
            title = f"{parent}: {name}" if parent else name
    flush()
    return passages


def _bare(text: str) -> list[str]:
    """The words of a text, without the punctuation around them ("foot." and "foot" match)."""
    return [word.strip(".,;:!?\"'()“”‘’") for word in text.split()]


def where_is(chunks: list[str], phrase: str) -> None:
    """Say which chunks contain a phrase, or show where it was cut in two."""
    norm = " ".join(phrase.split()).lower()
    flat = [" ".join(c.split()).lower() for c in chunks]
    found = [i + 1 for i, c in enumerate(flat) if norm and norm in c]
    if found:
        label = "chunk" if len(found) == 1 else "chunks"
        print(f'"{phrase}" is in {label} {", ".join(map(str, found))}.')
        return
    words = _bare(norm)
    for i in range(len(flat) - 1):
        left, right = _bare(flat[i]), _bare(flat[i + 1])
        for j in range(len(words) - 1, 0, -1):
            if left[-j:] == words[:j] and right[: len(words) - j] == words[j:]:
                print(f'"{phrase}" was cut in two, between chunk {i + 1} and chunk {i + 2}.')
                print(f"Chunk {i + 1} ends with: ...{chunks[i][-80:]}")
                print(f"Chunk {i + 2} starts with: {chunks[i + 1][:80]}...")
                return
    print(f'"{phrase}" is not in any chunk.')


def top_passages(question: str, passages: list[str], top: int = 3) -> list[tuple[float, str]]:
    """The `top` passages closest in meaning to the question, best first, as (score, passage)."""
    if not passages:
        return []
    vecs = _embed([question] + list(passages))
    return [(s, passages[i]) for i, s in rank(vecs[0], vecs[1:], top)]


def search(question: str, passages: list[str], top: int = 3) -> None:
    """Print the passages closest to the question: rank, score, start of the text."""
    if MODELS is None:
        print(NOT_READY)
        return
    for n, (s, passage) in enumerate(top_passages(question, passages, top), start=1):
        print(f"{n}. {s:.2f}  {_short(passage)}")


def _stream(prompt: str) -> None:
    for piece in MODELS.generator.generate(prompt):
        print(piece, end="", flush=True)
    print()


def ask(question: str) -> None:
    """Ask the AI with no documents."""
    if MODELS is None:
        print(NOT_READY)
        return
    print("Answer:")
    _stream(question)


def ask_with_documents(question: str, passages: list[str], top: int = 3, say_dont_know: bool = True) -> None:
    """Find the best passages, show them, and ask the AI to answer from them."""
    if MODELS is None:
        print(NOT_READY)
        return
    contexts = [p for _, p in top_passages(question, passages, top)]
    print("Passages sent to the AI:")
    for n, c in enumerate(contexts, start=1):
        print(f"{n}. {_short(c)}")
    print()
    print("Answer:")
    _stream(build_prompt(question, contexts, say_dont_know))


print("Loading the two AI models. This takes 1 to 2 minutes...")
MODELS = load_models()
print(status_line(MODELS.device))


## 1. The problem (3 min)

The notebook uses one document: a shortened copy of the Wikipedia article about the 2026 FIFA World Cup final, played on July 19, 2026. Spain beat Argentina 1–0 after extra time. The AI model in this notebook was trained before that day, so it has never read about this match.

**Do this:**
1. Run the cell below. It asks the AI how long the halftime break lasted in the final. The article says over 27 minutes, because of a halftime show.
2. Read the answer. Is it right?
3. Change the question to "Who won the 2026 FIFA World Cup final?" and run the cell again. Keep the quotes.

In [ ]:
# Change the question and run the cell again.
ask("How long did the halftime break last in the 2026 World Cup final?")

The AI has never read about this final. It may say so, or it may make up an answer that sounds right.

## 2. Embeddings: comparing meanings (22 min)

To compare meanings, the computer turns each sentence into a list of 384 numbers, called an embedding. Sentences with similar meanings get similar lists. The similarity score compares two lists: close to 0 means unrelated, and the closer to 1, the closer the meanings.

**Do this:**
1. The cell below compares "Spain won the final." with three other sentences. Guess which one gets the highest score, then run the cell.
2. Look at the score of the second sentence. It shares the word "final" with the first one, but its score is low.

In [ ]:
similarity("Spain won the final.", "The Spanish team became world champions.")
similarity("Spain won the final.", "This is my final offer.")
similarity("Spain won the final.", "I had soup for lunch.")

**Do this:**
1. Guess the score of "I love winter." and "I hate winter.", then run the cell.
2. Write two sentences of your own between the quotes and run the cell again.

In [ ]:
# Change the two sentences and run the cell again.
similarity("I love winter.", "I hate winter.")

The score tells you whether two sentences talk about the same thing, not whether they agree.

## 3. Search: finding the right passage (15 min)

Search uses the same scores. It compares your question with every passage and lists the closest ones first, with their scores. Here a passage is one section of our shortened copy of the article. There are 8: Introduction, Background, Route to the final: Spain, Route to the final: Argentina, Match: First half, Match: Second half, Match: Extra time, and Post-match.

**Do this:**
1. Guess which section will come first for "Which awards did Spain's players win?", then run the cell.

In [ ]:
passages = sections(ARTICLE)
search("Which awards did Spain's players win?", passages)

**Do this:**
1. Run the cell below. The referee's name, Slavko Vinčić, is in the Introduction. Is the Introduction in the list?

In [ ]:
passages = sections(ARTICLE)
search("Who was the referee of the final?", passages)

**Do this:**
1. Run the cell below. No section is about pizza. Look at the scores.
2. Change the question to one of your own and run the cell again.

In [ ]:
# Change the question and run the cell again.
passages = sections(ARTICLE)
search("What is the best pizza topping?", passages)

Search always returns passages, even when none of them answers the question, and the best one is not always first.

## 4. Splitting: cutting the article into chunks (15 min)

The embedding model reads only about the first 200 words of a passage and ignores the rest. Even those 200 words talk about many things, so the embedding is a mix of them. The referee is one sentence in the Introduction, so the Introduction does not stand out for that question. The fix is to cut the article into short pieces, called chunks, each with its own embedding.

**Do this:**
1. Run the cell below. It cuts the article into chunks of 50 words and searches them. Is the referee's name in the first result?
2. Change 50 to 5 and run the cell again. The first results are bits of sentences, and none has the name.
3. Change it to 0 and run the cell again. 0 means one chunk for the whole article. Its score is lower: the model reads only the first 200 words of that chunk, and they talk about many things.

In [ ]:
# Change 50 to 5, then to 0, and run the cell again each time.
chunks = split(ARTICLE, words_per_chunk=50)
search("Who was the referee of the final?", chunks)

A chunk can also end in the middle of an answer.

**Do this:**
1. Run the cell below. It looks for the sentence that says why a goal by Williams was disallowed. That sentence was cut in two: it starts at the end of one chunk and ends at the start of the next.

In [ ]:
chunks = split(ARTICLE, words_per_chunk=50)
where_is(chunks, "the goal was disallowed because Merino had stepped on Otamendi's foot")

Overlap fixes this: each chunk starts with the last words of the chunk before it.

**Do this:**
1. Run the cell below. With an overlap of 15 words, the sentence is whole again, in one chunk.
2. Change 15 to 0 and run the cell again.

In [ ]:
# Change 15 to 0 and run the cell again.
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
where_is(chunks, "the goal was disallowed because Merino had stepped on Otamendi's foot")

Long chunks mix many topics, and short chunks, or chunks cut in the wrong place, lose the answer.

## 5. Answering: the AI reads the chunks (15 min)

Now the AI gets the chunks that search finds. Each cell below cuts the article as in section 4 (50 words, overlap 15), then sends the AI one message: an instruction, the 3 best chunks and your question. The instruction says to answer from the chunks, and to say "I don't know based on these documents." when the answer is not there.

**Do this:**
1. Run the cell below. It asks about the halftime break again, this time with the article. It first shows the chunks it sent, under "Passages sent to the AI".
2. Compare the answer with the one you got in section 1.

In [ ]:
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
ask_with_documents("How long did the halftime break last?", chunks)

**Do this:**
1. Run the cell below. With top=1, the AI gets only the best chunk, and that chunk does not say where the final was played.
2. Change top=1 to top=3 and run the cell again.

In [ ]:
# Change top=1 to top=3 and run the cell again.
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
ask_with_documents("Where was the final played?", chunks, top=1)

**Do this:**
1. Run the cell below. Our copy of the article says there was a halftime show, but we removed the part that says who performed.
2. Change True to False and run the cell again. This removes the "I don't know" instruction. Compare the two answers.

In [ ]:
# Change True to False and run the cell again.
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
ask_with_documents("Who performed at the halftime show?", chunks, say_dont_know=True)

**Do this:**
1. Run the cell below. It asks why the goal by Williams was disallowed, with the overlap of 15 words from section 4.
2. Change 15 to 0 and run the cell again. The sentence is cut in two again. Look at the chunks sent to the AI: is the reason still there?

In [ ]:
# Change 15 to 0 and run the cell again.
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
ask_with_documents("Why was the goal by Williams disallowed?", chunks)

The AI can only answer as well as the chunks it gets.

## 6. Wrap-up (15 min)

RAG has four steps:
1. Cut the documents into chunks.
2. Compute an embedding for each chunk.
3. For each question, find the chunks closest to it.
4. Send those chunks and the question to the AI, which writes the answer.

**Do this:**
1. Run the cell below. It asks the referee question from section 3 again. Its two lines do all four steps.
2. Change the question to one of your own about the final and run the cell again.

In [ ]:
# Change the question and run the cell again.
chunks = split(ARTICLE, words_per_chunk=50, overlap=15)
ask_with_documents("Who was the referee of the final?", chunks)

Each choice can change the answers: the chunk size, the overlap, the number of chunks sent and the instruction.

### Running it on your own computer

In Colab, your text goes to Google's computers. On your own computer, your documents stay with you. Ollama, the app from Part 2, runs AI models on your own computer. This notebook also runs in Jupyter on a laptop.

**Do this:**
1. Click **File > Download > Download .ipynb** to save this notebook.
2. Install Python 3.11 or newer.
3. In a terminal, run `pip install sentence-transformers transformers torch accelerate jupyterlab`, then `jupyter lab`.
4. In Jupyter, open the file you saved.

### Sources

- The article: "2026 FIFA World Cup final", Wikipedia, [revision 1385698804](https://en.wikipedia.org/w/index.php?title=2026_FIFA_World_Cup_final&oldid=1385698804), retrieved 2026-11-20. License: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Shortened: some sections removed.
- The models: all-MiniLM-L6-v2 for the embeddings, and Qwen3-1.7B (with a GPU) or Qwen3-0.6B (without one) for the answers.

## Bonus: your own text file

You can ask questions about your own text. In Colab, the file you upload goes to Google's computers, so only upload text you are allowed to share.

**Do this:**
1. Save your document as a plain text file. In Google Docs, click **File > Download > Plain text (.txt)**. In Word, click **File > Save As** and choose **Plain Text**.
2. Run the first cell below. Click **Choose Files** under the cell (in some browsers it says **Browse**) and pick your file.
3. In the second cell, replace the question with your own. Keep the quotes. Then run the cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
my_text = list(uploaded.values())[0].decode("utf-8", errors="replace")

In [ ]:
# Change the question and run the cell again.
my_chunks = split(my_text, words_per_chunk=50, overlap=15)
ask_with_documents("Type your question here", my_chunks)

Not in Colab? Put your file next to the notebook and replace the first cell with this line, with your file's name:

```
my_text = open("my_file.txt", encoding="utf-8").read()
```

The same four steps work on any text, though this embedding model searches English text best.